# FraudMind Version 1 Demo

Two agents, one graph.

The ATO Agent and Payment Agent are wired into a LangGraph `StateGraph` with shared `FraudState`. Each case passes through both nodes sequentially and both verdicts are printed.

**Graph flow:** `START → ato_node → payment_node → END`

| Case | Session | Payment | Expected |
|---|---|---|---|
| 1 | Clear ATO (new device, Lagos login, email change) | Fraudulent (28x avg, 8 tx/hr, new account) | ATO: block / Payment: block |
| 2 | Clean login (known device, home geo) | Legitimate (in-store, normal amount) | ATO: allow / Payment: allow |
| 3 | Ambiguous login (new device, prior travel location) | Suspicious velocity (gaming, 6 tx/hr) | ATO: step_up / Payment: escalate or step_up |

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph.fraud_graph import fraud_graph
from src.schemas.ato_schemas import MockATOSignals
from src.schemas.payment_schemas import PaymentSignals

In [ ]:
def print_result(label: str, result: dict) -> None:
    """Pretty-print the output of a graph invocation."""
    print(f"{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")

    ato = result.get('ato_verdict')
    if ato:
        print(f"\nATO VERDICT")
        print(f"  verdict:    {ato.verdict.upper()}")
        print(f"  confidence: {ato.confidence:.0%}")
        print(f"  signals:    {', '.join(ato.primary_signals)}")
        print(f"  reasoning:  {ato.reasoning}")
        print(f"  action:     {ato.recommended_action}")

    payment = result.get('payment_verdict')
    if payment:
        print(f"\nPAYMENT VERDICT")
        print(f"  verdict:    {payment.verdict.upper()}")
        print(f"  confidence: {payment.confidence:.0%}")
        print(f"  signals:    {', '.join(payment.primary_signals)}")
        print(f"  reasoning:  {payment.reasoning}")
        print(f"  action:     {payment.recommended_action}")

    print()

## Case 1: Clear ATO + Fraudulent Payment

Expected: ATO **block**, Payment **block**

In [ ]:
case1_state = {
    "account_id": "acc_001",
    "ato_signals": MockATOSignals(
        account_id="acc_001",
        account_age_days=210,
        is_new_device=True,
        device_id="dev_unknown_001",
        login_geo="Lagos, Nigeria",
        account_home_geo="San Jose, CA",
        geo_mismatch=True,
        time_since_last_login_days=2,
        immediate_high_value_action=True,
        email_change_attempted=True,
        password_change_attempted=False,
        support_ticket_language_match=False,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_001",
        account_id="acc_001",
        amount_usd=2400.00,
        merchant_category="electronics",
        merchant_country="Nigeria",
        account_home_country="United States",
        is_international=True,
        transactions_last_1h=8,
        transactions_last_24h=14,
        avg_transaction_amount_30d=85.00,
        amount_vs_avg_ratio=28.2,
        is_first_transaction=False,
        card_present=False,
        days_since_account_creation=3,
    ),
}

result1 = fraud_graph.invoke(case1_state)
print_result("Case 1: Clear ATO + Fraudulent Payment", result1)

## Case 2: Clean Login + Legitimate Payment

Expected: ATO **allow**, Payment **allow**

In [ ]:
case2_state = {
    "account_id": "acc_002",
    "ato_signals": MockATOSignals(
        account_id="acc_002",
        account_age_days=847,
        is_new_device=False,
        device_id="dev_trusted_047",
        login_geo="San Jose, CA",
        account_home_geo="San Jose, CA",
        geo_mismatch=False,
        time_since_last_login_days=1,
        immediate_high_value_action=False,
        email_change_attempted=False,
        password_change_attempted=False,
        support_ticket_language_match=True,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_002",
        account_id="acc_002",
        amount_usd=94.50,
        merchant_category="grocery",
        merchant_country="United States",
        account_home_country="United States",
        is_international=False,
        transactions_last_1h=1,
        transactions_last_24h=3,
        avg_transaction_amount_30d=88.00,
        amount_vs_avg_ratio=1.07,
        is_first_transaction=False,
        card_present=True,
        days_since_account_creation=847,
    ),
}

result2 = fraud_graph.invoke(case2_state)
print_result("Case 2: Clean Login + Legitimate Payment", result2)

## Case 3: Ambiguous Login + Suspicious Velocity

Expected: ATO **step_up**, Payment **escalate** or **step_up**

In [ ]:
case3_state = {
    "account_id": "acc_003",
    "ato_signals": MockATOSignals(
        account_id="acc_003",
        account_age_days=430,
        is_new_device=True,
        device_id="dev_unknown_002",
        login_geo="London, UK",
        account_home_geo="New York, NY",
        geo_mismatch=True,
        time_since_last_login_days=12,
        immediate_high_value_action=False,
        email_change_attempted=False,
        password_change_attempted=False,
        support_ticket_language_match=True,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_003",
        account_id="acc_003",
        amount_usd=310.00,
        merchant_category="gaming",
        merchant_country="United Kingdom",
        account_home_country="United States",
        is_international=True,
        transactions_last_1h=6,
        transactions_last_24h=9,
        avg_transaction_amount_30d=120.00,
        amount_vs_avg_ratio=2.58,
        is_first_transaction=False,
        card_present=False,
        days_since_account_creation=430,
    ),
}

result3 = fraud_graph.invoke(case3_state)
print_result("Case 3: Ambiguous Login + Suspicious Velocity", result3)